# Lab 08 Pre-Lab: Control Theory Basics in Code

**Computational Sensorimotor Control** | Week 8 | Before Thursday's lab

---

## Overview

Section C of Lab 08 closes a feedback loop through vision, and finds that the delay
limits the gains. The **Control Theory Basics** handout explains the tools behind that
section: transfer functions, poles, and P and PD control. This notebook runs the
handout's algebra in Python, so that Thursday's lab starts from numbers you have already
seen.

**Before you start.** Read the whole handout, Boards 1–11.

| task | handout | what you do |
|---|---|---|
| **P.1** | Boards 1–4 | find the cart's poles, and read their time constants off its motion |
| **P.2** | Boards 5–9 | close the loop on the cart with P, PD, PI and PID control, and read the new poles |
| **P.3** | Board 10 | give the hand Lab 08's PD gains, without a delay |
| **P.4** | Board 11 | add vision's 155 ms delay, with and without the D term |

**How to work on it.** About 25 minutes, on your own or with your group. It needs only
NumPy, SciPy and Matplotlib, so it runs in Colab or any Jupyter installation without the
course package. Each task has one line to complete (P.2 has two), marked `# your code
here`, and a TEST cell that checks it. Run the cells in order.

---
## Setup

Run the next cell. `step_metrics` measures a step response the way the handout does: the
10–90% rise time, the overshoot, and the time after which the response stays within 2%
of its final value.

In [ ]:
# === Setup (provided — nothing to write) ===
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from scipy.special import lambertw

plt.rcParams.update({'figure.figsize': (10, 3.6), 'font.size': 11, 'axes.grid': True, 'grid.alpha': 0.3})
NAVY, TEAL, ORANGE, RED, GRAY = '#1B2A4A', '#2E86AB', '#E67E22', '#E74C3C', '#7F8C8D'

m, b, k = 1.0, 10.0, 20.0               # the cart of Board 1: kg, N s/m, N/m
TT = np.linspace(0, 12, 120001)         # 12 s in steps of 0.1 ms

def step_metrics(sys_, final):
    "Step response of a transfer function, with its 10-90% rise time, overshoot (%) and 2% settling time."
    t, y = signal.step(sys_, T=TT)
    i10, i90 = np.argmax(y >= 0.1 * final), np.argmax(y >= 0.9 * final)
    outside = np.nonzero(np.abs(y - final) > 0.02 * abs(final))[0]
    settle = 0.0 if len(outside) == 0 else (t[outside[-1] + 1] if outside[-1] + 1 < len(t) else np.inf)   # inf: never settles
    return dict(t=t, y=y, rise=t[i90] - t[i10], overshoot=max(0.0, 100 * (y.max() - final) / final), settle=settle)

print('ready')

---
## Task P.1 — The Cart's Poles

> 📘 **Control Theory Basics:** Boards 1–4.
>
> Board 3 gives the cart's transfer function from force to position,
> $G(s) = 1/(m s^2 + b s + k)$. Its poles are the roots of the denominator, which is
> Board 1's characteristic equation (Equation H.2). By Board 4, a real pole $p = -a$ adds
> a part $e^{-at}$ to the movement, with the time constant $\tau = 1/a$. After four time
> constants, 2% of that part is left.
>
> The cell pushes the cart with a steady 1 N, as in Board 1, and marks two times on its
> response. Complete one line: the time constant of each pole.

In [ ]:
G = signal.TransferFunction([1.0], [m, b, k])
poles = np.sort(G.poles.real)                   # 1/s; both of the cart's poles are real
# TODO: COMPLETE (1 line) — the time constant of each pole, in s (Board 4)
tau = ...  # your code here
cart = step_metrics(G, 1 / k)                   # the response to a steady 1 N push, from rest
print(f'poles {poles[0]:.2f} and {poles[1]:.2f} per s;  time constants {tau[0]:.3f} s and {tau[1]:.3f} s')
print(f'four time constants of the slow pole: {4 * tau.max():.2f} s;  within 2% of F/k after {cart["settle"]:.2f} s')

plt.figure(figsize=(6, 3.4))
plt.plot(cart['t'], 100 * cart['y'], color=NAVY, label='the cart, 1 N push')
plt.axhline(100 / k, color=GRAY, ls='--', label='F/k = 5 cm')
plt.axvline(4 * tau.max(), color=TEAL, ls=':', label='four time constants of the slow pole')
plt.axvline(cart['settle'], color=ORANGE, ls=':', label='within 2% from here on')
plt.xlim(0, 3); plt.xlabel('time (s)'); plt.ylabel('position (cm)'); plt.legend(fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
# ============================================================
#  TEST — the cart's poles  (Task P.1)
#  Strategy: the poles are -5 ± √5 (the quadratic formula on
#  s² + 10 s + 20). A time constant is a positive time, 1/a for the
#  pole -a; the handout's are 0.14 and 0.36 s.
# ============================================================
assert np.allclose(poles, [-7.23606797749979, -2.76393202250021]), 'the cart has poles at -5 ± √5 per s'
assert np.all(tau > 0), 'a time constant is a positive time: 1/a for the pole -a'
assert np.allclose(np.sort(tau), [0.1381966011250105, 0.3618033988749895]), \
    'tau = 1/a: expected 0.138 s and 0.362 s'
assert abs(cart['settle'] - 1.5894) < 1e-3, 'the cart settles within 2% after 1.59 s (Board 1)'
print('P.1 OK')

### Question P.1 — reading the poles

1. Which of the two poles decides when the movement ends, and why?
2. Four time constants of the slow pole come to 1.45 s, but the cart stays within 2% of its final 5 cm only after 1.59 s. Use Equation H.4 to explain the difference.

*Your answer:*

---
## Task P.2 — P, PD, PI and PID Control of the Cart

> 📘 **Control Theory Basics:** Boards 5–9.
>
> A controller sets the force from the error $e = r - x$ between a target $r$ and the
> position $x$. Board 5 gives the loop from target to position, $T = CG/(1 + CG)$, and the
> PID controller $C(s) = K_p + K_i/s + K_d s$ (Equation H.14). Clearing the fractions gives
> two forms, written here with $m$ kept:
>
> - without the integral term (P and PD control; Equation H.16):
>   $$T(s) = \frac{K_d s + K_p}{m s^2 + (b + K_d)\,s + (k + K_p)}$$
> - with it (PI and PID control; Equations H.17 and H.18):
>   $$T(s) = \frac{K_d s^2 + K_p s + K_i}{m s^3 + (b + K_d)\,s^2 + (k + K_p)\,s + K_i}$$
>
> The cell runs CTMS's four controllers toward a target 1 m away: P ($K_p$ = 300 N/m), PD
> (300 N/m and $K_d$ = 10 N s/m), PI (30 N/m and $K_i$ = 70 N/(m s)), and PID (350, 300 and
> 50). Complete two lines: the two denominators, as lists of coefficients from the highest
> power of $s$ down.

In [ ]:
def closed_loop(Kp, Ki=0.0, Kd=0.0):
    "T(s) from target to position, for the cart under C(s) = Kp + Ki/s + Kd s (Board 5)."
    if Ki == 0:                                     # P and PD (Equation H.16)
        num = [Kd, Kp] if Kd else [Kp]
        # TODO: COMPLETE (1 line) — the denominator m s^2 + (b + Kd) s + (k + Kp), as a list
        den = ...  # your code here
    else:                                           # PI and PID (Equations H.17 and H.18)
        num = [Kd, Kp, Ki] if Kd else [Kp, Ki]
        # TODO: COMPLETE (1 line) — the denominator m s^3 + (b + Kd) s^2 + (k + Kp) s + Ki, as a list
        den = ...  # your code here
    return signal.TransferFunction(num, den)

def pole_text(p):
    "The poles as text: the real ones, then each complex pair once."
    real = [f'{z.real:.2f}' for z in sorted(p, key=lambda z: z.real) if abs(z.imag) < 1e-9]
    pairs = [f'{z.real:.2f} ± {z.imag:.2f}i' for z in p if z.imag > 1e-9]
    return ', '.join(real + pairs)

GAINS = {'P': (300.0, 0.0, 0.0), 'PD': (300.0, 0.0, 10.0),          # (Kp, Ki, Kd): N/m, N/(m s), N s/m
         'PI': (30.0, 70.0, 0.0), 'PID': (350.0, 300.0, 50.0)}
COLORS = {'P': ORANGE, 'PD': TEAL, 'PI': RED, 'PID': NAVY}
loops = {}
fig, ax = plt.subplots(1, 2)
for name, (Kp, Ki, Kd) in GAINS.items():
    T_ = closed_loop(Kp, Ki, Kd)
    T0 = T_.num[-1] / T_.den[-1]                    # the final value, T(0)
    loops[name] = dict(poles=T_.poles, T0=T0, **step_metrics(T_, T0))
    print(f'{name:3s}: poles {pole_text(T_.poles)} per s;  overshoot {loops[name]["overshoot"]:.1f}%;  '
          f'within 2% after {loops[name]["settle"]:.2f} s;  T(0) = {T0:.4f}, steady-state error {100 * (1 - T0):.2f}%')
    a = ax[0] if Ki == 0 else ax[1]
    a.plot(loops[name]['t'], loops[name]['y'], color=COLORS[name], label=name)
for a, title in zip(ax, ('A   P and PD', 'B   PI and PID')):
    a.axhline(1, color=GRAY, ls='--', label='target')
    a.set_xlim(0, 2); a.set_xlabel('time (s)'); a.set_ylabel('position (m)')
    a.legend(fontsize=8); a.set_title(title, loc='left')
plt.tight_layout(); plt.show()

In [ ]:
# ============================================================
#  TEST — the four controllers  (Task P.2)
#  Strategy: T(0) and the poles must match Boards 6–9, and the
#  overshoot the handout's Tables 5 and 6. K_d moves the poles away
#  from the vertical axis and leaves T(0) where it was; K_i makes
#  T(0) = K_i/K_i = 1.
# ============================================================
REF = {
    'P': (0.9375, [complex(-5.000000, -17.175564), complex(-5.000000, 17.175564)], 40.070),
    'PD': (0.9375, [complex(-10.000000, -14.832397), complex(-10.000000, 14.832397)], 15.348),
    'PI': (1.0, [complex(-3.953787, -4.221558), complex(-3.953787, 4.221558), complex(-2.092426, 0.000000)], 1.261),
    'PID': (1.0, [complex(-53.144007, 0.000000), complex(-5.899054, 0.000000), complex(-0.956940, 0.000000)], 0.000),
}     # T(0), poles (1/s), overshoot (%): the handout's results
BOARD = {'P': 'Board 6', 'PD': 'Board 7: K_d adds to the damping', 'PI': 'Board 8', 'PID': 'Board 9'}
TEXT = {'P': '-5.00 ± 17.18i', 'PD': '-10.00 ± 14.83i', 'PI': '-2.09, -3.95 ± 4.22i', 'PID': '-53.14, -5.90, -0.96'}
for name, (T0, poles_ref, os_ref) in REF.items():
    L_ = loops[name]
    if name in ('P', 'PD'):
        assert abs(L_['T0'] - T0) < 1e-9, \
            f'{name}: T(0) should be Kp / (k + Kp) = 300/320; the cart\'s spring k stays in the constant term'
    else:
        assert abs(L_['T0'] - 1) < 1e-9, f'{name}: with the integral term T(0) = Ki / Ki = 1; the last coefficient is Ki'
    assert np.allclose(np.sort_complex(L_['poles']), np.sort_complex(poles_ref), atol=1e-6), \
        f'{name}: the poles should be {TEXT[name]} ({BOARD[name]})'
    assert abs(L_['overshoot'] - os_ref) < 0.1, f'{name}: overshoot {L_["overshoot"]:.1f}%, the handout gives {os_ref:.1f}%'
print('P.2 OK')

### Question P.2 — what each term does

1. Read the P loop's poles as in Board 4: how many times per second does the cart swing, and what is the time constant of the shrinking swings?
2. Which part of the poles did $K_d$ change, and what did that do to the response?
3. Which controllers end at the target, and why? Lab 08's loop has no integral term: why does the hand not need one (Board 10)?
4. The PID response has no overshoot, but its last percent is slow. Which of its poles is responsible, and what is its time constant?

*Your answer:*

---
## Task P.3 — The Hand Under Lab 08's Gains

> 📘 **Control Theory Basics:** Board 10.
>
> The hand in Lab 08 has no spring and no damper. Week 6's inverse dynamics makes it
> accelerate as commanded, so $G(s) = 1/s^2$ along every axis, and a PD command
> $a = K_p e + K_d \dot e$ gives the denominator $s^2 + K_d s + K_p$. Lab 08 sets the gains
> through $\kappa = K_p\Delta^2$ and $\delta = K_d\Delta$, with $\Delta$ = 155 ms for central
> vision (notes §3.2). This task takes the gains of the notes' Figure 4, $\kappa$ = 0.2 and
> $\delta$ = 0.9, and leaves the delay out.
>
> The hand starts 1 cm from a still target. Transforming $\ddot e + K_d \dot e + K_p e = 0$
> with $e(0)$ = 1 cm and $\dot e(0) = 0$ (Equation H.6, twice) gives
> $E(s) = e(0)\,(s + K_d)/(s^2 + K_d s + K_p)$, so `signal.impulse` returns $e(t)$.
>
> Complete one line: the damping ratio $\zeta$ of $s^2 + K_d s + K_p$. Match it to
> $s^2 + 2\zeta\omega_n s + \omega_n^2$, as in Board 6.

In [ ]:
DELAY = 0.155                                   # central vision's effective delay, s (notes §3.2)
kappa, delta = 0.2, 0.9                         # the gains of the notes' Figure 4 and Lab Task C.2
Kp, Kd = kappa / DELAY**2, delta / DELAY        # 1/s^2 and 1/s: gains per unit mass (Board 10)
wn = np.sqrt(Kp)
# TODO: COMPLETE (1 line) — the damping ratio of s^2 + Kd s + Kp
zeta = ...  # your code here
E0 = 0.01                                       # the starting offset, m
t3 = np.linspace(0, 4, 40001)                   # 4 s in steps of 0.1 ms
_, e_free = signal.impulse(signal.TransferFunction([E0, E0 * Kd], [1, Kd, Kp]), T=t3)
print(f'Kp = {Kp:.2f} /s^2, Kd = {Kd:.2f} /s:  wn = {wn:.2f} rad/s,  zeta = {zeta:.3f}')
print(f'poles of s^2 + Kd s + Kp: {", ".join(f"{p:.2f}" for p in np.sort(np.roots([1, Kd, Kp]).real))} per s')
print(f'error after 0.5 s {100*e_free[5000]:.2f} cm, after 1 s {100*e_free[10000]:.3f} cm; smallest {100*e_free.min():.4f} cm')

plt.figure(figsize=(6, 3.4))
plt.plot(t3, 100 * e_free, color=NAVY, label='no delay')
plt.axhline(0, color=GRAY, lw=0.8)
plt.xlabel('time (s)'); plt.ylabel('error (cm)'); plt.legend(fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
# ============================================================
#  TEST — the hand without a delay  (Task P.3)
#  Strategy: matching s² + K_d s + K_p to s² + 2ζω_n s + ω_n² gives
#  ζ = K_d / (2ω_n). With ζ just above 1, both poles are real and the
#  error falls to zero without crossing it.
# ============================================================
assert abs(zeta - 1.0062305899) < 1e-9, 'zeta = Kd / (2 wn), with wn = sqrt(Kp)'
assert np.all(np.abs(np.roots([1, Kd, Kp]).imag) < 1e-12), 'with zeta above 1 both poles are real (Board 4)'
assert e_free.min() > 0, 'without the delay the error should fall to zero without crossing it'
print('P.3 OK')

---
## Task P.4 — Add the Delay

> 📘 **Control Theory Basics:** Board 11.
>
> Vision reports the hand $\Delta$ late, so the controller acts on the error as it was
> $\Delta$ ago: $\ddot e(t) = -K_p\,e(t-\Delta) - K_d\,\dot e(t-\Delta)$. Board 11 turns this into
> $s^2 + (K_p + K_d s)\,e^{-s\Delta} = 0$ (Equation H.20). That is not a polynomial, so
> `signal` has no coefficients to work with. Instead, `run_loop` steps the equation forward
> 0.1 ms at a time and keeps the history of $e$ and $\dot e$. Before $t = 0$ the hand sat
> still at the offset, so until the delay has passed the controller sees step 0.
>
> The cell runs three loops: PD without the delay (it must match Task P.3), PD with the
> 155 ms delay, and the delayed loop without the D term. The dashed line is the growth
> that the notes' Equation 8.5 predicts for the last one.
>
> Complete one line: the step whose error the controller sees, `lag` steps back and never
> before step 0. Lab 08's Task C.1 writes the same line for the arm.

In [ ]:
def run_loop(Kp, Kd, delay, e0=E0, t_end=4.0, dt=1e-4):
    "Error (m) and its rate (m/s) under e'' = -Kp e(t - delay) - Kd e'(t - delay), from rest at e0."
    n, lag = int(round(t_end / dt)), int(round(delay / dt))
    e, v = np.zeros(n + 1), np.zeros(n + 1)
    e[0] = e0
    for i in range(n):
        # TODO: COMPLETE (1 line) — the step the controller sees: lag steps back, not before step 0
        j = ...  # your code here
        v[i + 1] = v[i] + (-Kp * e[j] - Kd * v[j]) * dt
        e[i + 1] = e[i] + v[i + 1] * dt
    return np.arange(n + 1) * dt, e, v

t4, e_now, _ = run_loop(Kp, Kd, 0.0)                    # PD, no delay
_, e_late, v_late = run_loop(Kp, Kd, DELAY)             # PD, 155 ms delay
t8, e_noD, _ = run_loop(Kp, 0.0, DELAY, t_end=8.0)      # no D term, 155 ms delay
# the delayed PD error stalls where its rate comes back up to a peak near zero
stalls = [i for i in range(1, len(v_late) - 1) if v_late[i] > v_late[i-1] and v_late[i] >= v_late[i+1] and t4[i] > 0.3]
s_P = complex(2 / DELAY * lambertw(1j * np.sqrt(Kp) * DELAY / 2))     # notes Equation 8.5 (Lab Task C.3)
a_ = np.abs(e_noD)
peaks = [i for i in range(1, len(a_) - 1) if a_[i] >= a_[i-1] and a_[i] > a_[i+1] and t8[i] > 1.0]
growth = np.polyfit(t8[peaks], np.log(a_[peaks]), 1)[0]

print(f'no delay: largest difference from Task P.3, {100*np.abs(e_now - e_free).max():.5f} cm')
print(f'155 ms delay: the error stalls at {", ".join(f"{t4[i]:.2f}" for i in stalls[:4])} s,'
      f' at {", ".join(f"{100*e_late[i]:.3f}" for i in stalls[:4])} cm; smallest {100*e_late.min():.4f} cm')
print(f'no D term: the error grows {growth:.2f} per s; Equation 8.5 predicts {s_P.real:.2f} per s at {s_P.imag:.2f} rad/s')

fig, ax = plt.subplots(1, 2)
ax[0].plot(t4, 100 * e_now, color=NAVY, label='PD, no delay')
ax[0].plot(t4, 100 * e_late, color=RED, label='PD, 155 ms delay')
ax[0].plot(t4[stalls[:4]], 100 * e_late[stalls[:4]], 'o', mfc='none', mec=RED, label='stalls')
ax[0].axhline(0, color=GRAY, lw=0.8); ax[0].set_xlabel('time (s)'); ax[0].set_ylabel('error (cm)')
ax[0].legend(fontsize=8); ax[0].set_title('A   with the D term', loc='left')
ax[1].semilogy(t8, 100 * a_, color=ORANGE, label='no D term, 155 ms delay')
ax[1].semilogy(t8, 100 * E0 * np.exp(s_P.real * t8), '--', color=NAVY, label='Equation 8.5')
ax[1].set_xlabel('time (s)'); ax[1].set_ylabel('|error| (cm)'); ax[1].set_ylim(1e-2, 1e3)
ax[1].legend(fontsize=8); ax[1].set_title('B   without it', loc='left')
plt.tight_layout(); plt.show()

In [ ]:
# ============================================================
#  TEST — the delayed loop  (Task P.4)
#  Strategy: (1) with no delay, run_loop must reproduce Task P.3's
#  exact solution; (2) with the delay, the PD error falls in steps
#  without crossing zero, stalls about once every 0.86 s, and shrinks
#  at the rate of Board 11's root; (3) without the D term the error
#  grows at the rate of the notes' Equation 8.5.
# ============================================================
STALL_T = [0.8068, 1.6682, 2.5297, 3.3913]       # s (the handout's Figure 7B)
ROOT = (-1.860541, 7.285520)         # Board 11: the rightmost root, 1/s
assert np.abs(e_now - e_free).max() < 1e-5, 'with delay = 0, run_loop should match Task P.3 within 0.001 cm'
assert len(stalls) >= 4, 'with the delay the error should stall (stop, then fall again) about once every 0.86 s; check j'
assert np.allclose(t4[stalls[:4]], STALL_T, atol=2e-3), f'the error should stall at {STALL_T} s; check that j is lag steps back'
assert e_late.min() > 0, 'with the delay the PD error falls in steps but never crosses zero'
decay = np.log(e_late[stalls[0]] / e_late[stalls[2]]) / (t4[stalls[2]] - t4[stalls[0]])
assert abs(decay / -ROOT[0] - 1) < 0.05, "the stalls should shrink at the rate of Board 11's root"
assert abs(growth / s_P.real - 1) < 0.03, 'without the D term the error should grow at the rate of Equation 8.5'
print('P.4 OK')

### Question P.3 — what the delay does

1. Board 11 gives the delayed loop's rightmost root as −1.86 ± 7.29i. Read it as in Board 4: how often does the error stall, and how quickly does it shrink? Compare with panel A of the figure in Task P.4.
2. Without the delay, position-only feedback would give $s^2 + K_p = 0$. What would the hand do (Board 4)? With the delay it grows (panel B). Use Check 11 of Board 11 to explain why.
3. Lab 08's Task C.5 finds the largest $\kappa = K_p\Delta^2$ for which the delayed loop can be stable with the best $K_d$. Write down your guess before the lab.

*Your answer:*

---
## Bring to Thursday

Write these down, on paper or in your group's notes. Section C of Lab 08 uses each of
them.

| from | what to bring |
|---|---|
| P.1 | the time constant of the cart's slow pole |
| P.2 | how many times per second the P loop swings, the PD loop's overshoot, and which controllers end at the target |
| P.3 | $\zeta$ of the hand loop with $\kappa$ = 0.2 and $\delta$ = 0.9, without the delay |
| P.4 | how often the delayed PD error stalls, how fast the loop without the D term grows, and your guess for the largest stable $\kappa$ |

In Lab 08, Task C.1 builds this loop on the two-joint arm, and Task C.2 feeds it a plan
read at the wrong time. Task C.3 measures Task P.4's growth on the arm, and Task C.5 finds
the largest stable gains.

### AI-use disclosure

Per the syllabus, AI assistance is **not permitted** on labs. If you used any AI tool
while working on this notebook, describe what you used and how here.

*Your disclosure:*